# Phase 2.2 Experiment Runner

**实验内容：**
1. Phase 2.1b Alignment (9 runs, ~9 hours)
2. Loss-scale Diagnostics (3 runs, ~1 hour)
3. Gamma 精调 (12 runs, ~8 hours)

**总计：** 24 runs, ~18 hours

In [ ]:
# 1. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
%%bash
set -euo pipefail

# 2. Clone or update repo
if [ ! -d "/content/FYP/.git" ]; then
  git clone https://github.com/ROUCHER27/FYP.git /content/FYP
  cd /content/FYP
  pip install -q -r requirements.txt
fi

cd /content/FYP
git fetch origin
git checkout phase2-fixes
git pull origin phase2-fixes

echo "Branch: $(git branch --show-current)"
echo "Commit: $(git rev-parse --short HEAD)"

In [ ]:
%%bash
# 3. Verify Drive paths
mkdir -p /content/drive/MyDrive/FYP/phase2_1b/results
mkdir -p /content/drive/MyDrive/FYP/phase2_1b/checkpoints
mkdir -p /content/drive/MyDrive/FYP/phase2_1b/logs
mkdir -p /content/drive/MyDrive/FYP/phase2_diagnostics
mkdir -p /content/drive/MyDrive/FYP/phase2_gamma/results
mkdir -p /content/drive/MyDrive/FYP/phase2_gamma/checkpoints
mkdir -p /content/drive/MyDrive/FYP/phase2_gamma/logs

echo "✅ Drive paths ready"

## Part 1: Phase 2.1b Alignment (9 runs, ~9 hours)

验证 Phase 2 runner 与 Phase 1.5 结果一致

In [ ]:
%%bash
set -euo pipefail

cd /content/FYP

python run_phase2_1b_alignment.py \
  --losses imadl,gmadl,hybrid_mul \
  --seeds 42,52,62 \
  --caps 0.05 \
  --data-dir /content/FYP \
  --test-months 24 \
  --max-epochs 20 \
  --batch-size 1024 \
  --output-root /content/drive/MyDrive/FYP/phase2_1b/results \
  --checkpoint-root /content/drive/MyDrive/FYP/phase2_1b/checkpoints \
  --log-root /content/drive/MyDrive/FYP/phase2_1b/logs \
  --skip-existing \
  --resume-mode auto \
  2>&1 | tee /content/drive/MyDrive/FYP/phase2_1b/logs/alignment_$(date +%Y%m%d_%H%M%S).log

echo "✅ Phase 2.1b Alignment complete"

## Part 2: Loss-scale Diagnostics (3 runs, ~1 hour)

检查 top 3 losses 的分量量级平衡

In [ ]:
%%bash
set -euo pipefail

cd /content/FYP

python run_loss_scale_diagnostics.py \
  --losses imadl_m2_alpha06,m2_robust_gamma01,m2_robust_gamma10 \
  --seed 42 \
  --data-dir /content/FYP \
  --test-months 24 \
  --max-epochs 5 \
  --batch-size 1024 \
  --output-dir /content/drive/MyDrive/FYP/phase2_diagnostics \
  2>&1 | tee /content/drive/MyDrive/FYP/phase2_diagnostics/diagnostics_$(date +%Y%m%d_%H%M%S).log

echo "✅ Loss-scale Diagnostics complete"

## Part 3: Gamma 精调 (12 runs, ~8 hours)

测试 gamma=0.3, 0.5, 0.7, 1.5 找最优值

In [ ]:
%%bash
set -euo pipefail

cd /content/FYP

LOSSES="m2_robust_gamma03,m2_robust_gamma05,m2_robust_gamma07,m2_robust_gamma15"

python run_phase2_gamma_refinement.py \
  --losses "${LOSSES}" \
  --seeds 42,52,62 \
  --caps 0.05 \
  --data-dir /content/FYP \
  --test-months 24 \
  --max-epochs 20 \
  --batch-size 1024 \
  --output-root /content/drive/MyDrive/FYP/phase2_gamma/results \
  --checkpoint-root /content/drive/MyDrive/FYP/phase2_gamma/checkpoints \
  --log-root /content/drive/MyDrive/FYP/phase2_gamma/logs \
  --skip-existing \
  --resume-mode auto \
  2>&1 | tee /content/drive/MyDrive/FYP/phase2_gamma/logs/gamma_refinement_$(date +%Y%m%d_%H%M%S).log

echo "✅ Gamma 精调 complete"

## Results Summary

查看各部分结果

In [ ]:
%%bash
echo "=== Phase 2.1b Alignment Results ==="
find /content/drive/MyDrive/FYP/phase2_1b/results -name "sanity_summary_*.json" | wc -l
echo ""

echo "=== Loss-scale Diagnostics Results ==="
ls -lh /content/drive/MyDrive/FYP/phase2_diagnostics/*.json 2>/dev/null || echo "No results yet"
echo ""

echo "=== Gamma 精调 Results ==="
find /content/drive/MyDrive/FYP/phase2_gamma/results -name "sanity_summary_*.json" | wc -l